# Retrieval-Augmented Generation (RAG)

In [1]:
from setup import bedrock_tool
import chromadb
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


Create a static calorie table that we can use as a tool:

In [13]:
# We populated the RAG with the data from the data/calories.csv file in
# the ../rag_setup/rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_qna")

In [14]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):   
    print(doc)
    print("\n")

Question: What is the recommended amount of bananas I should consume to count it as a single serving?
        Answer: One small-sized banana can be counted as a single serving.

        This Q&A pair provides information about nutrition and health topics.


Question: Which food is recommended for infants after they've been introduced to ripe bananas and sweet potatoes?
        Answer: Introduce porridge made from wheat flour or ground rice, starting with only one cereal. Once a week has passed, you may increase the frequency of this new food to two feedings per day.

        This Q&A pair provides information about nutrition and health topics.




In [6]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
print(calorie_lookup_tool('bananas'))

SyntaxError: invalid syntax (2993000333.py, line 2)

In [15]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(calorie_lookup_tool.__dict__)],
)

In [ ]:
with trace("Nutrition Assistant with RAG") as t:
    result = await Runner.run(
        calorie_agent,
        "To be healthy, what is the right nutriotion we should get per a day?",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 

A healthy diet typically includes:

- **Calories**: 2,000-2,500 calories per day for an average adult.
- **Macronutrients**:
  - **Carbohydrates**: 45-65% of total daily calories.
  - **Proteins**: 10-35% of total daily calories.
  - **Fats**: 20-35% of total daily calories.
- **Micronutrients**:
  - **Vitamins**: Various, including A, B, C, D, E, and K.
  - **Minerals**: Including calcium, iron, potassium, and magnesium.
- **Fiber**: About 25-30 grams per day.
- **Water**: About 2-3 liters per day for adults.

These values can vary based on age, sex, weight, height, and level of physical activity.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_0ff7825812b549b1aa57672b8f006fb2
